# Практическое занятие 15. Сбор текстового корпуса на узбекском языке, его анализ и представление результатов исследования

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/p15.ipynb)

**Цели занятия:**

- Пройти полный цикл корпусного исследования: сбор → очистка → анализ → визуализация → отчёт
- Собрать с веб-сайта корпус узбекских текстов с метаданными (категория, дата, автор)
- Рассчитать корпусные показатели: токены, словарь, TTR, статистика по категориям, частотные слова
- Сохранить корпус в форматах JSON Lines и CSV и представить результаты в виде отчёта на Markdown

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Корпусное исследование как конвейер

**Текстовый корпус** — собранная по определённым правилам коллекция текстов с **метаданными** (источник, дата, автор, жанр/категория). Корпус — основной инструмент компьютерной лингвистики: на нём изучают лексику, обучают модели, проверяют гипотезы. Для узбекского языка готовых корпусов мало, поэтому умение собрать свой корпус особенно важно.

| Этап | Что делаем | Инструменты |
|---|---|---|
| 1. Сбор | обход сайта, извлечение текстов и метаданных | `requests`, `BeautifulSoup` |
| 2. Очистка | удаление рекламы и навигации, нормализация апострофов и регистра | `re`, `str` |
| 3. Токенизация | разбиение на слова | `re.findall(r"\w+", ...)` |
| 4. Анализ | объём, словарь, частоты, сравнение категорий | `pandas`, `Counter` |
| 5. Визуализация | диаграммы частот и объёмов | `matplotlib` |
| 6. Хранение | корпус в стандартном формате | JSON Lines, CSV |
| 7. Отчёт | цель, данные, методы, результаты, выводы | Markdown |

## Требования к корпусу

- **Репрезентативность**: тексты разных категорий, авторов и дат, а не 100 статей одной рубрики.
- **Метаданные** у каждого документа: `url`, `title`, `category`, `date`, `author` — без них нельзя сравнивать подкорпуса.
- **Чистота**: только основной текст (`div.article-body p`), без рекламы `aside.ads`, меню и подвала.
- **Единая орфография**: узбекские апострофы приводятся к стандарту — oʻ, gʻ (U+02BB) и ʼ (U+02BC).
- **Документация**: откуда и когда собраны тексты, на каких условиях их можно использовать.

## Корпусные показатели

| Показатель | Определение | Как посчитать |
|---|---|---|
| Токены (tokens) | все словоупотребления | `len(tokens)` |
| Типы (types), словарь | разные слова | `len(set(tokens))` |
| TTR (type/token ratio) | лексическое разнообразие | `types / tokens` |
| Гапаксы | слова, встретившиеся 1 раз | `sum(1 for c in freq.values() if c == 1)` |
| Средняя длина документа | токенов на текст | `tokens / docs` |

> **Заметка: TTR зависит от объёма**  
> Чем больше текст, тем ниже TTR: частые слова повторяются, а новых становится всё меньше. Поэтому TTR сравнивают только для выборок одинакового размера.

## Статистика по подкорпусам

```python
stats = corpus.groupby("category").agg(docs=("url", "count"),
                                       tokens=("n_tokens", "sum"))
stats["avg_tokens"] = (stats["tokens"] / stats["docs"]).round(1)
stats.sort_values("tokens", ascending=False)
```

## Форматы хранения корпуса

| Формат | Плюсы | Минусы |
|---|---|---|
| CSV | открывается в Excel, компактный | неудобен для длинных текстов и списков |
| **JSON Lines** (`.jsonl`) | одна строка = один документ; дописывается построчно; читается потоково Spark, MongoDB, Hugging Face | не открывается в Excel |
| TXT + `meta.csv` | просто для ручного чтения | метаданные отдельно от текста |

```python
corpus.to_json("corpus.jsonl", orient="records", lines=True, force_ascii=False)
back = pd.read_json("corpus.jsonl", lines=True)
```

> **Внимание: force_ascii=False**  
> По умолчанию pandas записывает не-ASCII символы как `\u02bb`. Файл остаётся корректным, но человек его не прочитает. С `force_ascii=False` узбекские буквы сохраняются как есть (в UTF-8).

## Отчёт об исследовании

Отчёт — это короткий документ, по которому коллега поймёт, что сделано и что получилось. Удобно собирать его автоматически из переменных с помощью f-строк, чтобы числа в отчёте всегда совпадали с расчётами:

```python
report = f"""# Корпус узбекских новостей
## Данные
- Документов: {len(corpus)}
- Токенов: {total_tokens}
## Результаты
- Самая крупная категория: {stats.index[0]}
"""
open("report.md", "w", encoding="utf-8").write(report)
```

Структура: **цель** → **данные** (источник, объём, период) → **методы** (очистка, токенизация, стоп-слова) → **результаты** (таблицы, графики, числа) → **выводы и ограничения** (маленький объём, одна редакция, неравные категории).

## Задание 1. Шаг 1. Список статей сайта

Начните сбор корпуса с учебного сайта `SITE`. Функция `get_soup(url)` уже определена.

Обойдите все страницы списка новостей, начиная с `SITE + "index.html"` и переходя по ссылке `a.next`:

- `links` — список **полных** адресов всех статей (16 штук, без повторов, в порядке появления);
- `n_pages` — сколько страниц списка вы посетили;
- между запросами — пауза `time.sleep(0.05)`.

<details><summary>💡 Подсказка</summary>

`links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]`; следующая страница: `nxt = soup.select_one("a.next")`, `url = urljoin(url, nxt["href"]) if nxt else None`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

In [ ]:
links = []
n_pages = 0
url = SITE + "index.html"

while url:
    soup = get_soup(url)
    n_pages += 1
    # добавьте в links полные адреса статей с этой страницы
    ...
    # перейдите на следующую страницу (или url = None)
    url = None
    time.sleep(0.05)

print(n_pages, "страницы,", len(links), "статей")
links[:3]

## Задание 2. Шаг 2. Корпус с метаданными

Список `links` (16 адресов) готов. Напишите функцию `parse_article(url)`, которая возвращает словарь:

| Ключ | Источник на странице статьи |
|---|---|
| `url` | адрес статьи |
| `title` | `h1.article-title` |
| `category`, `author` | `span.category`, `span.author` |
| `date` | атрибут `datetime` тега `time` |
| `views` | число из `span.views` (int) |
| `text` | абзацы `div.article-body p`, соединённые `"\n"` |

Соберите DataFrame `corpus` по всем статьям (с паузой `time.sleep(0.05)`) и выведите число документов и символов.

<details><summary>💡 Подсказка</summary>

Внутри функции: `s.select_one("span.category").get_text(strip=True)`, `s.select_one("time")["datetime"]`, `int(re.search(r"\d+", s.select_one("span.views").get_text()).group())`, `"\n".join(p.get_text(strip=True) for p in s.select("div.article-body p"))`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

links, url = [], SITE + "index.html"
while url:
    soup = get_soup(url)
    links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None

In [ ]:
def parse_article(url):
    s = get_soup(url)
    return {
        "url": url,
        "title": s.select_one("h1.article-title").get_text(strip=True),
        "category": ...,
        "date": ...,
        "author": ...,
        "views": ...,
        "text": ...,
    }

rows = []
for u in links:
    rows.append(parse_article(u))
    time.sleep(0.05)

corpus = pd.DataFrame(rows)
print(len(corpus), "документов,", corpus["text"].str.len().sum(), "символов")
corpus.head(3)

## Задание 3. Шаг 3. Нормализация и токенизация

DataFrame `corpus` собран (на сайте апострофы типографские: ‘ ’).

1. Напишите `normalize(text)`: нижний регистр; все варианты апострофа (`'`, ‘, ’, обратный апостроф, ʻ, ʼ) после **o** и **g** → ʻ (U+02BB), в остальных местах → ʼ (U+02BC).
2. Напишите `tokenize(text)` → список слов: `re.findall(r"\w+", normalize(text))`, только буквенные (`w.isalpha()`).
3. Нормализуйте столбец `category` (ta’lim → taʼlim) и добавьте столбцы `tokens`, `n_tokens` (число токенов), `n_types` (число разных слов).
4. Посчитайте `total_tokens`, `vocab_size` (разных слов во всём корпусе) и `ttr = round(vocab_size / total_tokens, 3)`.

<details><summary>💡 Подсказка</summary>

В normalize: `for ch in ["'", "‘", "’", "\x60", "ʻ"]: text = text.replace(ch, "ʼ")`, затем `re.sub("([og])ʼ", "\\1ʻ", text)`. В tokenize добавьте фильтр `if w.isalpha()`. Счётчики: `corpus["tokens"].str.len()`, `corpus["tokens"].apply(lambda t: len(set(t)))`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

links, url = [], SITE + "index.html"
while url:
    soup = get_soup(url)
    links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None

def parse_article(url):
    s = get_soup(url)
    meta = s.select_one("div.meta")
    return {
        "url": url,
        "title": s.select_one("h1.article-title").get_text(strip=True),
        "category": meta.select_one("span.category").get_text(strip=True),
        "date": meta.select_one("time")["datetime"],
        "author": meta.select_one("span.author").get_text(strip=True),
        "views": int(re.search(r"\d+", meta.select_one("span.views").get_text()).group()),
        "text": "\n".join(p.get_text(strip=True) for p in s.select("div.article-body p")),
    }

corpus = pd.DataFrame([parse_article(u) for u in links])

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

In [ ]:
def normalize(text):
    text = text.lower()
    # 1) все апострофы -> "\u02bc";  2) после o/g -> "\u02bb"
    ...
    return text

def tokenize(text):
    return re.findall(r"\w+", normalize(text))

print(normalize("O‘zbekiston ta’limi: G'alaba!"))

corpus["category"] = corpus["category"].apply(normalize)
corpus["tokens"] = corpus["text"].apply(tokenize)
corpus["n_tokens"] = ...
corpus["n_types"] = ...

total_tokens = ...
vocab_size = ...
ttr = ...
print(total_tokens, vocab_size, ttr)
corpus[["title", "category", "n_tokens", "n_types"]]

## Задание 4. Шаг 4. Статистика по категориям

В `corpus` есть нормализованные `category`, `tokens`, `n_tokens`, `n_types`.

Постройте таблицу `stats` (индекс — категория) со столбцами:

- `docs` — число документов;
- `tokens` — сумма токенов;
- `types` — число **разных** слов во всех текстах категории;
- `avg_tokens` — средняя длина документа в токенах (округлить до 1 знака).

Отсортируйте по `tokens` по убыванию. Какая категория крупнейшая? Почему сравнивать TTR категорий здесь некорректно?

<details><summary>💡 Подсказка</summary>

`tokens=("n_tokens", "sum")`; разные слова: `corpus.groupby("category")["tokens"].apply(lambda s: len(set(w for t in s for w in t)))`; `(stats["tokens"] / stats["docs"]).round(1)`; `sort_values("tokens", ascending=False)`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

links, url = [], SITE + "index.html"
while url:
    soup = get_soup(url)
    links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None

def parse_article(url):
    s = get_soup(url)
    meta = s.select_one("div.meta")
    return {
        "url": url,
        "title": s.select_one("h1.article-title").get_text(strip=True),
        "category": meta.select_one("span.category").get_text(strip=True),
        "date": meta.select_one("time")["datetime"],
        "author": meta.select_one("span.author").get_text(strip=True),
        "views": int(re.search(r"\d+", meta.select_one("span.views").get_text()).group()),
        "text": "\n".join(p.get_text(strip=True) for p in s.select("div.article-body p")),
    }

corpus = pd.DataFrame([parse_article(u) for u in links])

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

corpus["category"] = corpus["category"].apply(normalize)
corpus["tokens"] = corpus["text"].apply(tokenize)
corpus["n_tokens"] = corpus["tokens"].str.len()
corpus["n_types"] = corpus["tokens"].apply(lambda t: len(set(t)))
total_tokens = int(corpus["n_tokens"].sum())
vocab_size = len(set(w for t in corpus["tokens"] for w in t))

In [ ]:
stats = corpus.groupby("category").agg(
    docs=("url", "count"),
    tokens=...,
)
stats["types"] = ...
stats["avg_tokens"] = ...
stats = stats.sort_values(...)
stats

## Задание 5. Шаг 5. Частотные слова и графики

1. `top_words` — 15 самых частых **содержательных** слов корпуса (не из `stop`, длина ≥ 3): список пар `(слово, частота)`.
2. Рисунок 1×2:
   - слева — горизонтальная диаграмма `top_words` (самое частое слово вверху);
   - справа — столбчатая диаграмма числа токенов по категориям (`stats["tokens"]`).
3. Заголовки у обоих графиков, подписи осей с частотами/токенами. Сохраните рисунок в файл `corpus_charts.png` (`fig.savefig`).

<details><summary>💡 Подсказка</summary>

`content.update(w for w in toks if w not in stop and len(w) >= 3)`; слева: `words, counts = zip(*top_words[::-1])`, `axes[0].barh(words, counts)`; справа: `stats["tokens"].plot(kind="bar", ax=axes[1])`; `fig.savefig("corpus_charts.png", dpi=150, bbox_inches="tight")` — до `plt.show()`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

links, url = [], SITE + "index.html"
while url:
    soup = get_soup(url)
    links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None

def parse_article(url):
    s = get_soup(url)
    meta = s.select_one("div.meta")
    return {
        "url": url,
        "title": s.select_one("h1.article-title").get_text(strip=True),
        "category": meta.select_one("span.category").get_text(strip=True),
        "date": meta.select_one("time")["datetime"],
        "author": meta.select_one("span.author").get_text(strip=True),
        "views": int(re.search(r"\d+", meta.select_one("span.views").get_text()).group()),
        "text": "\n".join(p.get_text(strip=True) for p in s.select("div.article-body p")),
    }

corpus = pd.DataFrame([parse_article(u) for u in links])

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

corpus["category"] = corpus["category"].apply(normalize)
corpus["tokens"] = corpus["text"].apply(tokenize)
corpus["n_tokens"] = corpus["tokens"].str.len()
corpus["n_types"] = corpus["tokens"].apply(lambda t: len(set(t)))
total_tokens = int(corpus["n_tokens"].sum())
vocab_size = len(set(w for t in corpus["tokens"] for w in t))

stats = corpus.groupby("category").agg(docs=("url", "count"), tokens=("n_tokens", "sum")).sort_values("tokens", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

content = Counter()
for toks in corpus["tokens"]:
    ...
top_words = content.most_common(15)
print(top_words[:5])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# слева: barh по top_words
# справа: stats["tokens"].plot(kind="bar", ax=axes[1])
plt.tight_layout()
plt.show()

## Задание 6. Шаг 6. Сохранение корпуса: JSON Lines и CSV

Сохраните корпус в двух форматах:

1. `corpus_uz.jsonl` — JSON Lines (`orient="records", lines=True`), **узбекские символы без экранирования** (`force_ascii=False`). Сохраните все столбцы, кроме `tokens`.
2. `corpus_uz.csv` — CSV без индекса, кодировка UTF-8, тоже без `tokens`.
3. Прочитайте JSON Lines обратно в `back` и убедитесь, что тексты совпадают. Выведите первую строку файла.

<details><summary>💡 Подсказка</summary>

`out.to_json("corpus_uz.jsonl", orient="records", lines=True, force_ascii=False)`; `out.to_csv("corpus_uz.csv", index=False, encoding="utf-8")`; `back = pd.read_json("corpus_uz.jsonl", lines=True)`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

links, url = [], SITE + "index.html"
while url:
    soup = get_soup(url)
    links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None

def parse_article(url):
    s = get_soup(url)
    meta = s.select_one("div.meta")
    return {
        "url": url,
        "title": s.select_one("h1.article-title").get_text(strip=True),
        "category": meta.select_one("span.category").get_text(strip=True),
        "date": meta.select_one("time")["datetime"],
        "author": meta.select_one("span.author").get_text(strip=True),
        "views": int(re.search(r"\d+", meta.select_one("span.views").get_text()).group()),
        "text": "\n".join(p.get_text(strip=True) for p in s.select("div.article-body p")),
    }

corpus = pd.DataFrame([parse_article(u) for u in links])

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

corpus["category"] = corpus["category"].apply(normalize)
corpus["tokens"] = corpus["text"].apply(tokenize)
corpus["n_tokens"] = corpus["tokens"].str.len()
corpus["n_types"] = corpus["tokens"].apply(lambda t: len(set(t)))
total_tokens = int(corpus["n_tokens"].sum())
vocab_size = len(set(w for t in corpus["tokens"] for w in t))

In [ ]:
out = corpus.drop(columns=["tokens"])

out.to_json("corpus_uz.jsonl", orient="records", lines=True)
# сохраните CSV

back = ...
print(len(back), "документов прочитано")
print(open("corpus_uz.jsonl", encoding="utf-8").readline()[:300])

## Задание 7. Шаг 7. Отчёт об исследовании

Все результаты готовы: `corpus`, `total_tokens`, `vocab_size`, `ttr`, `stats` (по категориям, по убыванию токенов), `top_words` (15 пар «слово, частота»).

Сформируйте строку `report` в формате Markdown (через f-строку), чтобы числа подставлялись из переменных:

- заголовок первого уровня (`# ...`);
- не меньше трёх разделов второго уровня (`## Данные`, `## Результаты`, `## Выводы`);
- в отчёте: число документов и категорий, `total_tokens`, `vocab_size`, `ttr`, крупнейшая категория (`stats.index[0]`) и **топ-3 слова**;
- не меньше трёх пунктов списка (`- ...`).

Сохраните отчёт в `report.md` и выведите его.

<details><summary>💡 Подсказка</summary>

Подставляйте значения в f-строку: `{len(corpus)}`, `{corpus['category'].nunique()}`, `{total_tokens}`, `{vocab_size}`, `{ttr}`, `{stats.index[0]}`, `{top3}`. Сохранение: `open("report.md", "w", encoding="utf-8").write(report)`.

</details>

In [ ]:
# Подготовительный код задания
import re, time, json, os
from collections import Counter
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    """Скачать страницу (UTF-8) и вернуть BeautifulSoup."""
    r = requests.get(url)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

links, url = [], SITE + "index.html"
while url:
    soup = get_soup(url)
    links += [urljoin(url, a["href"]) for a in soup.select("h2.news-title a")]
    nxt = soup.select_one("div.pagination a.next")
    url = urljoin(url, nxt["href"]) if nxt else None

def parse_article(url):
    s = get_soup(url)
    meta = s.select_one("div.meta")
    return {
        "url": url,
        "title": s.select_one("h1.article-title").get_text(strip=True),
        "category": meta.select_one("span.category").get_text(strip=True),
        "date": meta.select_one("time")["datetime"],
        "author": meta.select_one("span.author").get_text(strip=True),
        "views": int(re.search(r"\d+", meta.select_one("span.views").get_text()).group()),
        "text": "\n".join(p.get_text(strip=True) for p in s.select("div.article-body p")),
    }

corpus = pd.DataFrame([parse_article(u) for u in links])

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

corpus["category"] = corpus["category"].apply(normalize)
corpus["tokens"] = corpus["text"].apply(tokenize)
corpus["n_tokens"] = corpus["tokens"].str.len()
corpus["n_types"] = corpus["tokens"].apply(lambda t: len(set(t)))
total_tokens = int(corpus["n_tokens"].sum())
vocab_size = len(set(w for t in corpus["tokens"] for w in t))

stats = corpus.groupby("category").agg(docs=("url", "count"), tokens=("n_tokens", "sum"))
stats["types"] = corpus.groupby("category")["tokens"].apply(lambda s: len(set(w for t in s for w in t)))
stats["avg_tokens"] = (stats["tokens"] / stats["docs"]).round(1)
stats = stats.sort_values("tokens", ascending=False)
content = Counter(w for t in corpus["tokens"] for w in t if w not in stop and len(w) >= 3)
top_words = content.most_common(15)

ttr = round(vocab_size / total_tokens, 3)

In [ ]:
top3 = ", ".join(w for w, _ in top_words[:3])

report = f"""# Корпус ...

## Данные
- Источник: учебный сайт «Talaba Yangiliklari»
- Документов: ...

## Результаты
...

## Выводы
...
"""

print(report)

## Самостоятельная работа

## Самостоятельная работа

1. Расширьте корпус: добавьте тексты из `data/corpus_uz/` и `data/news_raw.csv` (после очистки), объедините с собранными статьями без дубликатов и пересчитайте показатели.
2. Посчитайте число гапаксов и постройте кривую роста словаря (число разных слов в зависимости от числа токенов) — закон Хипса.
3. Подготовьте доклад на 5 минут по теме «Перспективы применения технологий Big Data в компьютерной лингвистике» на примере вашего корпуса: какие задачи (поиск, классификация, языковые модели) можно решать на корпусе в 1000 раз большем.